## 1.Download the datasets

### 1.1 Download the multi-omics data

* parse the multiomics data from the synapse website of the dataset ROSMAP https://www.synapse.org/#!Synapse:syn23446022
* Genome Variants https://www.synapse.org/#!Synapse:syn26263118
* Methylation https://www.synapse.org/#!Synapse:syn3168763
* RNA sequence https://www.synapse.org/#!Synapse:syn3505720
* Proteomics https://www.synapse.org/#!Synapse:syn21266454    

### 1.2 Download the clinical data

* parse the clinical data https://www.synapse.org/#!Synapse:syn3191087

## 2.Read the files

In [ ]:
import pandas as pd
from pyensembl import EnsemblRelease

### 2.1 Read DNA methylation data

In [ ]:
methylation_value = pd.read_csv("./ROSMAP-raw/Methylation/DNA-methylation-array/AMP-AD_ROSMAP_Rush-Broad_IlluminaHumanMethylation450_740_imputed.tsv",sep='\t')

In [ ]:
methylation_value

### 2.2 Read Platform annotations for 450k methylation 

In [ ]:
# Reading and processing basic data
try:
    annotation = pd.read_table('./ROSMAP-raw/Methylation/DNA-methylation-array/GPL16304-47833.txt', delimiter='\t')
    annotation['Distance_closest_TSS'] = annotation['Distance_closest_TSS'].astype(int)
    annotation = annotation[~annotation['Closest_TSS'].apply(lambda x: len(str(x).split(';')) > 1)]
except ValueError as e:
    print(f"Unable to convert 'Closest_TSS' column to integer: {e}")
    problematic_rows = annotation['Distance_closest_TSS'].apply(lambda x: not str(x).isnumeric())
    print("Problematic rows:")
    print(annotation.loc[problematic_rows])

annotation

In [ ]:
map_annotation= annotation[['ID', 'Closest_TSS','Closest_TSS_gene_name', 'Distance_closest_TSS']]
map_annotation

### 2.3 Read gene expression data, mapping information and substitue the the gene name

In [ ]:
gene_expression = pd.read_csv('./ROSMAP-raw/RNASeq/ROSMAP_RNAseq_FPKM_gene.tsv', sep='\t')
gene_expression

In [ ]:
ensembl_gene_ids = gene_expression['gene_id'].apply(lambda x: x.split('.')[0]).tolist()
gene_expression['gene_id'] = ensembl_gene_ids
gene_expression

In [ ]:
ensembl_data_unique_gene = pd.read_csv("./ROSMAP-raw/Meta-Data/mart_export_unique_gene.txt")
ensembl_data_unique_gene

In [ ]:
ensembl_data = pd.read_csv("./ROSMAP-raw/Meta-Data/mart_export_genename.txt")
ensembl_data= ensembl_data.rename(columns={'Gene stable ID': 'gene_id'})
ensembl_data

In [ ]:
ensembl_data = ensembl_data.dropna()
ensembl_data = ensembl_data.drop_duplicates()
ensembl_data

In [ ]:
# Now, merge the two dataframes on the 'gene_name' column
merged_data = pd.merge(gene_expression, ensembl_data, on='gene_id', how='inner')
merged_data

In [ ]:
merged_data = merged_data.rename(columns={'Gene name': 'gene_name'})
merged_data.replace('Nan', pd.NA, inplace=True)  # Replace 'Nan' string with actual NaN
merged_data.dropna(subset=['gene_name'], inplace=True)  # Drop rows with NaN values
merged_data

In [ ]:
import numpy as np
cols = ['gene_name'] + [col for col in merged_data if col != 'gene_name']
gene_expression_new = merged_data[cols]
numeric_cols = gene_expression_new.select_dtypes(include=[np.number]).columns.tolist()

# Now, we group by 'gene_name' and calculate the mean of all numeric columns
gene_expression_new = gene_expression_new.groupby('gene_name')[numeric_cols].mean()

# Reset the index to turn the grouped keys (gene_name) back into a column
gene_expression_new.reset_index(inplace=True)
gene_expression_new

### 2.4 Read clinical data

In [ ]:
survival = pd.read_csv('./ROSMAP-raw/ROSMAP_clinical/ROSMAP_clinical.csv', sep=',')
survival

### 2.5 Read proteomics data

In [ ]:
protein = pd.read_csv("./ROSMAP-raw/Proteomics/C2.median_polish_corrected_log2(abundanceRatioCenteredOnMedianOfBatchMediansPerProtein)-8817x400.csv",sep=',')
protein

In [ ]:
protein.iloc[:, 0] = protein.iloc[:, 0].str.split('|').str[0]
protein.columns = ['gene_name'] + protein.columns[1:].tolist()

In [ ]:
#calculate the NaN proportion of each row
nan_proportions = protein.isna().mean(axis=1)
# Display the results
print(nan_proportions)

In [ ]:
# Identify gene names that have more than one occurrence
duplicated_genes = protein['gene_name'][protein['gene_name'].duplicated(keep=False)]
print(duplicated_genes.shape)
# Display the unique gene names that were averaged
unique_duplicated_genes = duplicated_genes.unique()
print(unique_duplicated_genes.shape)
unique_duplicated_genes

In [ ]:
# Fill NaN values with 0 in the remaining rows
# protein = protein[nan_proportions <= 1/3]
protein = protein.groupby('gene_name', as_index=False).mean()
protein = protein.fillna(0)
protein

### 2.6 Read Genome Variants data

In [ ]:
cnv_data = pd.read_csv('./ROSMAP-raw/GenomeVariants/ROSMAP.CNV.Matrix.txt',sep='\t')
cnv_data

In [ ]:
ensembl_data = EnsemblRelease()
ensembl_data.download()
ensembl_data.index()

In [ ]:
# use pyensemb to map the gene name
def get_gene_names(chromosome, start, end):
    genes = ensembl_data.genes_at_locus(contig=chromosome, position=start, end=end)
    gene_names = [gene.gene_name for gene in genes]
    return ', '.join(gene_names) if gene_names else None

# Apply the function to each row in the DataFrame
cnv_data['gene_name'] = cnv_data.apply(
    lambda row: get_gene_names(str(row['CHROM']), row['START'], row['END']), axis=1
)
cnv_data

In [ ]:
# Function to clean gene names
def clean_gene_names(gene_names):
    if gene_names:
        # Split the string by comma, strip whitespace and periods, then filter out empty strings
        genes = [gene.strip(' .') for gene in gene_names.split(',') if gene.strip(' .')]
        # Join the cleaned gene names with a comma
        return ', '.join(genes)
    else:
        # If gene_names is None or empty, return a placeholder
        return 'No gene found'

# Apply the function to clean up the gene names
cnv_data['gene_name'] = cnv_data['gene_name'].apply(clean_gene_names)
cnv_data

In [ ]:
cnv_data['gene_name'] = cnv_data['gene_name'].str.split(', ')
cnv_data_exploded = cnv_data.explode('gene_name')
cnv_data_exploded

In [ ]:
# Filter out rows with empty 'Gene_Names' or 'No gene found'
cnv_data_exploded = cnv_data_exploded[(cnv_data_exploded['gene_name'] != '') & (cnv_data_exploded['gene_name'] != 'No gene found')]

sample_columns = cnv_data_exploded.columns[6:-1] 
cnv_aggregated = cnv_data_exploded.groupby(['gene_name','SVTYPE'])[sample_columns].sum().reset_index()
# cnv_aggregated = cnv_data_exploded.groupby('gene_name')[sample_columns].sum().reset_index()
cnv_aggregated

In [ ]:
cnv_aggregated_del = cnv_aggregated[cnv_aggregated['SVTYPE'] == 'DEL']
cnv_aggregated_dup = cnv_aggregated[cnv_aggregated['SVTYPE'] == 'DUP']
cnv_aggregated_mcnv = cnv_aggregated[cnv_aggregated['SVTYPE'] == 'mCNV']
print(f"Shape of cnv_aggregated_del: {cnv_aggregated_del.shape}")
print(f"Shape of cnv_aggregated_dup: {cnv_aggregated_dup.shape}")
print(f"Shape of cnv_aggregated_mcnv: {cnv_aggregated_mcnv.shape}")

In [ ]:
# Create a DataFrame from the union set of all gene names
cnv_all_genes = pd.DataFrame({'gene_name': list(set(cnv_aggregated_del['gene_name']).union(set(cnv_aggregated_dup['gene_name']), set(cnv_aggregated_mcnv['gene_name'])))})

# Merge each aggregated DataFrame with the all genes DataFrame
# This will align all genes across all DataFrames, and fill missing entries with 0
cnv_aggregated_del = pd.merge(cnv_all_genes, cnv_aggregated_del, on='gene_name', how='outer').fillna(-1)
cnv_aggregated_dup = pd.merge(cnv_all_genes, cnv_aggregated_dup, on='gene_name', how='outer').fillna(-1)
cnv_aggregated_mcnv = pd.merge(cnv_all_genes, cnv_aggregated_mcnv, on='gene_name', how='outer').fillna(-1)

# Drop the 'SVTYPE' column from each merged DataFrame
cnv_aggregated_del.drop(columns='SVTYPE', inplace=True)
cnv_aggregated_dup.drop(columns='SVTYPE', inplace=True)
cnv_aggregated_mcnv.drop(columns='SVTYPE', inplace=True)
print(f"Shape of cnv_aggregated_del: {cnv_aggregated_del.shape}")
print(f"Shape of cnv_aggregated_dup: {cnv_aggregated_dup.shape}")
print(f"Shape of cnv_aggregated_mcnv: {cnv_aggregated_mcnv.shape}")

In [ ]:
cnv_aggregated_del

In [ ]:
# cnv_aggregated.iloc[:, 1:] = cnv_aggregated.iloc[:, 1:].where(cnv_aggregated.iloc[:, 1:] == 0, 1)
# cnv_aggregated

In [ ]:
# snphead = pd.read_csv('./ROSMAP-raw/GenomeVariants/SNP-Array-raw/AMP-AD_ROSMAP_Rush-Broad_AffymetrixGenechip6_Imputed.fam', sep='\t',header=None)
# snphead

In [ ]:
# survival_gwas_id = pd.DataFrame(survival.iloc[:, 1].astype(str) + survival.iloc[:, 0].astype(str), columns=['gwas_id'])
# survival_gwas_id['individualID'] = survival['individualID']

# # Displaying the new DataFrame
# survival_gwas_id

In [ ]:
# gwas_id_list = survival_gwas_id['gwas_id'].tolist()

# snphead['extracted_gwas_id'] = snphead.iloc[:, 0].apply(
#     lambda x: next((gwas_id for gwas_id in gwas_id_list if gwas_id in x), np.nan)
# )

# # Perform a left merge with 'survival_gwas_id' on the extracted 'gwas_id' to keep all rows from 'snphead'
# # and to get the corresponding 'individualID' where there is a match
# merged_result = pd.merge(
#     snphead, 
#     survival_gwas_id, 
#     how='left', 
#     left_on='extracted_gwas_id', 
#     right_on='gwas_id'
# )

# # Select only the 'gwas_id' and 'individualID' columns for the final result
# # Fill missing values with 'NA' to indicate unmatched rows
# snphead_updated = merged_result[['gwas_id', 'individualID']].fillna('NA')

# # Display the head of the final DataFrame
# snphead_updated

In [ ]:
# snphead_updated_withoutNA = snphead_updated[snphead_updated.iloc[:, 0] != "NA"]
# snphead_updated_withoutNA

## 3.Methylation data process

### 3.1 Define methylation region

In [ ]:
import pandas as pd
import numpy as np

#Define vectorized area determination function for methylation data
def vectorized_determine_region(distances):
    regions = ['Upstream', 'Distal Promoter', 'Proximal Promoter', 'Core Promoter', 'Downstream']
    conditions = [
        (-6000 <= distances) & (distances < -3000),
        (-3000 <= distances) & (distances < -250),
        (-250 <= distances) & (distances < -50),
        (-50 <= distances) & (distances <= 0),
        (0 < distances) & (distances <= 3000)
    ]
    return np.select(conditions, regions, default=None)

### 3.2 Merge the annotation files to the methylation data and apply region function

In [ ]:
# Merging basic data and methylation data
methylation_merged_df = pd.merge(map_annotation, methylation_value, left_on='ID', right_on='TargetID', how='right')

# Determining the region for each row outside the loop
methylation_merged_df['Region'] = vectorized_determine_region(methylation_merged_df['Distance_closest_TSS'])

methylation_merged_df = methylation_merged_df.dropna(subset=['Region'])  # Remove rows without a region

# Initializing a dictionary to store data for each region
regions_data = {region: pd.DataFrame() for region in ["Upstream", "Distal Promoter", "Proximal Promoter", "Core Promoter", "Downstream"]}

In [ ]:
methylation_merged_df

In [ ]:
original_gene_names = methylation_merged_df['Closest_TSS_gene_name'].copy()

# Use a regular expression to remove the period and any characters following it
methylation_merged_df['Closest_TSS_gene_name'] = methylation_merged_df['Closest_TSS_gene_name'].str.replace(r'\..*', '', regex=True)

# Determine how many rows were changed by comparing the new values to the original ones
rows_changed = (original_gene_names != methylation_merged_df['Closest_TSS_gene_name']).sum()

# Output the updated DataFrame and the number of rows that were changed
methylation_merged_df, rows_changed

In [ ]:
# Delete the 'sample' column
methylation_merged_df = methylation_merged_df.drop('TargetID', axis=1)
# Delete the 'ID' column
methylation_merged_df = methylation_merged_df.drop('ID', axis=1)
# Delete the 'Distance_closest_TSS' column
methylation_merged_df = methylation_merged_df.drop('Distance_closest_TSS', axis=1)

In [ ]:
methylation_merged_df

In [ ]:
methylation_merged_df['Closest_TSS'] = methylation_merged_df['Closest_TSS'].astype(int)
methylation_merged_df['Closest_TSS_gene_name'] = methylation_merged_df['Closest_TSS_gene_name'].astype(str)
methylation_merged_df['Region'] = methylation_merged_df['Region'].astype(str)
methylation_merged_df.to_csv('methylation_regions.csv', index=False)

In [ ]:
print(methylation_merged_df[['Closest_TSS', 'Closest_TSS_gene_name', 'Region']].dtypes)

### 3.3 Calculate the average methylation value of five regions

In [ ]:
# obtain all regions
regions = methylation_merged_df['Region'].unique()
regions

In [ ]:
import pandas as pd
# Initialize empty DataFrames for each region
Upstream_df = pd.DataFrame()
Distal_Promoter_df = pd.DataFrame()
Proximal_Promoter_df = pd.DataFrame()
Core_Promoter_df = pd.DataFrame()
Downstream_df = pd.DataFrame()

# Operate on each region
for region in regions:
    # Get all data for this region
    region_data = methylation_merged_df[methylation_merged_df['Region'] == region]
    
    # Group and calculate the average for each (TSS, Region) combination
    grouped = region_data.groupby(['Closest_TSS_gene_name', 'Region'], as_index=False).mean()
    
    # Since we split the data into different files based on Region, we can delete this column
    grouped = grouped.drop(columns=['Region'])
    
    # Print the shape of the grouped data
    print(f"Shape of {region}: {grouped.shape}")
    
    # Assign the grouped data to the respective DataFrame
    if region == 'Upstream':
        Upstream_df = grouped
    elif region == 'Distal Promoter':
        Distal_Promoter_df = grouped
    elif region == 'Proximal Promoter':
        Proximal_Promoter_df = grouped
    elif region == 'Core Promoter':
        Core_Promoter_df = grouped
    elif region == 'Downstream':
        Downstream_df = grouped

    # Optionally, save the data for this region to a new csv file
    # grouped.to_csv(f"{region}_averaged_tss_data.csv", index=False)


### 3.4 Unify gene and TSS for five methylation value files

In [ ]:
import pandas as pd

# from methylation files above DataFrame 
dfs = [Upstream_df, Distal_Promoter_df, Proximal_Promoter_df, Core_Promoter_df, Downstream_df]

# merge those files to find all combos 
all_genes_tss = pd.concat(dfs)['Closest_TSS_gene_name'].drop_duplicates()

In [ ]:
all_genes_tss

In [ ]:
# Merge unique combinations back into each DataFrame and fill NaN values with 0
# Upstream
Upstream_df = pd.merge(all_genes_tss, Upstream_df, on=['Closest_TSS_gene_name'], how='outer').fillna(0)
print(f"Shape of Upstream_df: {Upstream_df.shape}")

# Distal Promoter
Distal_Promoter_df = pd.merge(all_genes_tss, Distal_Promoter_df, on=['Closest_TSS_gene_name'], how='outer').fillna(0)
print(f"Shape of Distal_Promoter_df: {Distal_Promoter_df.shape}")

# Proximal Promoter
Proximal_Promoter_df = pd.merge(all_genes_tss, Proximal_Promoter_df, on=['Closest_TSS_gene_name'], how='outer').fillna(0)
print(f"Shape of Proximal_Promoter_df: {Proximal_Promoter_df.shape}")

# Core Promoter
Core_Promoter_df = pd.merge(all_genes_tss, Core_Promoter_df, on=['Closest_TSS_gene_name'], how='outer').fillna(0)
print(f"Shape of Core_Promoter_df: {Core_Promoter_df.shape}")

# Downstream
Downstream_df = pd.merge(all_genes_tss, Downstream_df, on=['Closest_TSS_gene_name'], how='outer').fillna(0)
print(f"Shape of Downstream_df: {Downstream_df.shape}")

In [ ]:
Upstream_df.rename(columns={'Closest_TSS_gene_name': 'gene_name'}, inplace=True)
Distal_Promoter_df.rename(columns={'Closest_TSS_gene_name': 'gene_name'}, inplace=True)
Proximal_Promoter_df.rename(columns={'Closest_TSS_gene_name': 'gene_name'}, inplace=True)
Core_Promoter_df.rename(columns={'Closest_TSS_gene_name': 'gene_name'}, inplace=True)
Downstream_df.rename(columns={'Closest_TSS_gene_name': 'gene_name'}, inplace=True)

display(Upstream_df)
display(Distal_Promoter_df)
display(Proximal_Promoter_df)
display(Core_Promoter_df)
display(Downstream_df)

## 4.Unify genes and patient samples within datasets

### 4.1 Unify gene and TSS for methylation, copynumer, and gene expression data

In [ ]:
gene_expression = gene_expression_new.copy()
gene_expression

In [ ]:
Upstream_df

In [ ]:
ensembl_data_unique_gene = pd.read_csv("./ROSMAP-raw/Meta-Data/mart_export_unique_gene.txt")
ensembl_data_unique_gene

In [ ]:
#filter the genes not in Ensembl Dataset
gene_expression = gene_expression[gene_expression['gene_name'].isin(ensembl_data_unique_gene['Gene name'])]
Upstream_df = Upstream_df[Upstream_df['gene_name'].isin(ensembl_data_unique_gene['Gene name'])]
protein = protein[protein['gene_name'].isin(ensembl_data_unique_gene['Gene name'])]
cnv_aggregated_del = cnv_aggregated_del[cnv_aggregated_del['gene_name'].isin(ensembl_data_unique_gene['Gene name'])]

In [ ]:
# Convert the gene name columns from each DataFrame to sets
gene_expression_genes = set(gene_expression['gene_name'])
print(f"Number of genes in gene_expression: {len(gene_expression_genes)}")
methylation_genes = set(Upstream_df['gene_name'])
print(f"Number of genes in methylation: {len(methylation_genes)}")
protein_genes = set(protein['gene_name'])
print(f"Number of genes in protein: {len(protein_genes)}")
cnv_aggregated_genes = set(cnv_aggregated_del['gene_name'])
print(f"Number of genes in cnv_aggregated: {len(cnv_aggregated_genes)}")

# Find the intersection of these sets 
# ????
common_genes =  gene_expression_genes | methylation_genes | protein_genes | cnv_aggregated_genes

# Convert the intersection back to a list, if needed
common_genes_list = list(common_genes)

# Print the number of common genes
print(f"Number of common genes: {len(common_genes)}")

In [ ]:
#count the Transcript type
ensembl_data_type = pd.read_csv("./ROSMAP-raw/Meta-Data/mart_export.txt")
ensembl_data_type= ensembl_data_type.rename(columns={'Gene name': 'gene_name'})
ensembl_data_type = ensembl_data_type.drop_duplicates(subset='gene_name', keep='first')

display(ensembl_data_type)

# Now, merge the two dataframes on the 'gene_name' column
gene_expression_transcript = pd.merge(gene_expression, ensembl_data_type, on='gene_name', how='left')
display(gene_expression_transcript)
protein_coding_count = (gene_expression_transcript['Transcript type'] == 'protein_coding').sum()
print(f'Number of rows with "protein_coding": {protein_coding_count}')

Upstream_df_transcript = pd.merge(Upstream_df, ensembl_data_type, on='gene_name', how='left')
display(Upstream_df_transcript)
protein_coding_count = (Upstream_df_transcript['Transcript type'] == 'protein_coding').sum()
print(f'Number of rows with "protein_coding": {protein_coding_count}')

protein_transcript = pd.merge(protein, ensembl_data_type, on='gene_name', how='left')
display(protein_transcript)
protein_coding_count = (protein_transcript['Transcript type'] == 'protein_coding').sum()
print(f'Number of rows with "protein_coding": {protein_coding_count}')

cnv_aggregated_del_transcript = pd.merge(cnv_aggregated_del, ensembl_data_type, on='gene_name', how='left')
display(cnv_aggregated_del_transcript)
protein_coding_count = (cnv_aggregated_del_transcript['Transcript type'] == 'protein_coding').sum()
print(f'Number of rows with "protein_coding": {protein_coding_count}')

In [ ]:
# common_genes_left = common_genes - common_genes_cnv_with_others
# print(f"Number of common genes: {len(common_genes_left)}")

In [ ]:
# cnv_aggregated_filtered = pd.DataFrame(columns=cnv_aggregated.columns)

# for gene in common_genes_cnv_with_others:
#     if gene in cnv_aggregated['Gene_Names'].values:
#         gene_row = cnv_aggregated[cnv_aggregated['Gene_Names'] == gene]
#         cnv_aggregated_filtered = pd.concat([cnv_aggregated_filtered, gene_row])

# for gene in common_genes_left:
#     zero_data = [0] * (len(cnv_aggregated.columns) - 1)
#     zero_row = pd.DataFrame([[gene] + zero_data], columns=cnv_aggregated.columns)
#     cnv_aggregated_filtered = pd.concat([cnv_aggregated_filtered, zero_row])

# cnv_aggregated_filtered.reset_index(drop=True, inplace=True)

In [ ]:
# cnv_aggregated_filtered

#### 4.1.1 Make the gene expression, mutation, copy number and proteomics data inputted

In [ ]:
common_genes_df = pd.DataFrame(common_genes, columns=['gene_name'])
common_genes_df

In [ ]:
merged_data_transcript_Upstream_df = pd.merge(common_genes_df, ensembl_data_type, on='gene_name', how='left')
protein_coding_count = (merged_data_transcript_Upstream_df['Transcript type'] == 'protein_coding').sum()
print(f'Number of rows with "protein_coding": {protein_coding_count}')

In [ ]:
unique_values = merged_data_transcript_Upstream_df['Transcript type'].value_counts()
unique_values

In [ ]:
gene_expression_inputted = pd.merge(common_genes_df ,gene_expression,on='gene_name',how='outer').fillna(0)
protein_inputted = pd.merge(common_genes_df, protein, on='gene_name', how='outer').fillna(0)

Upstream_df_inputted = pd.merge(common_genes_df, Upstream_df,on='gene_name',how='outer').fillna(0)
Distal_Promoter_df_inputted = pd.merge(common_genes_df, Distal_Promoter_df,on='gene_name',how='outer').fillna(0)
Proximal_Promoter_df_inputted = pd.merge(common_genes_df, Proximal_Promoter_df,on='gene_name',how='outer').fillna(0)
Core_Promoter_df_inputted = pd.merge(common_genes_df, Core_Promoter_df,on='gene_name',how='outer').fillna(0)
Downstream_df_inputted = pd.merge(common_genes_df, Downstream_df,on='gene_name',how='outer').fillna(0)

cnv_aggregated_inputted_del = pd.merge(common_genes_df, cnv_aggregated_del, on='gene_name', how='outer').fillna(-1)
cnv_aggregated_inputted_dup = pd.merge(common_genes_df, cnv_aggregated_dup, on='gene_name', how='outer').fillna(-1)
cnv_aggregated_inputted_mcnv = pd.merge(common_genes_df, cnv_aggregated_mcnv, on='gene_name', how='outer').fillna(-1)

gene_expression_inputted

#### 4.1.2  Intersecting genes with various databases

In [ ]:
import pandas as pd
# Add the gene names from databases like [KEGG / BioGRID] to intersect with the common genes
# KEGG
kegg_pathway_df = pd.read_csv('./Regulatory-network-data/KEGG/full_kegg_pathway_list.csv')
kegg_pathway_df = kegg_pathway_df[['source', 'target', 'pathway_name']]
kegg_df = kegg_pathway_df[kegg_pathway_df['pathway_name'].str.contains('signaling pathway|signaling pathways', case=False)]
print(kegg_df['pathway_name'].value_counts())
kegg_df = kegg_df.rename(columns={'source': 'src', 'target': 'dest'})
src_list = list(kegg_df['src'])
dest_list = list(kegg_df['dest'])
path_list = list(kegg_df['pathway_name'])
# ADJUST ALL GENES TO UPPERCASE
up_src_list = []
for src in src_list:
    up_src = src.upper()
    up_src_list.append(up_src)
up_dest_list = []
for dest in dest_list:
    up_dest = dest.upper()
    up_dest_list.append(up_dest)
up_kegg_conn_dict = {'src': up_src_list, 'dest': up_dest_list}
up_kegg_df = pd.DataFrame(up_kegg_conn_dict)
up_kegg_df = up_kegg_df.drop_duplicates()
up_kegg_df.to_csv('./Regulatory-network-data/KEGG/up_kegg.csv', index=False, header=True)
kegg_gene_list = list(set(list(up_kegg_df['src']) + list(up_kegg_df['dest'])))
print('----- NUMBER OF GENES IN KEGG: ' + str(len(kegg_gene_list)) + ' -----')
print(up_kegg_df.shape)

up_kegg_path_conn_dict = {'src': up_src_list, 'dest': up_dest_list, 'path': path_list}
up_kegg_path_df = pd.DataFrame(up_kegg_path_conn_dict)
up_kegg_path_df = up_kegg_path_df.drop_duplicates()
up_kegg_path_df.to_csv('./Regulatory-network-data/KEGG/up_kegg_path.csv', index=False, header=True)
kegg_path_gene_list = list(set(list(up_kegg_path_df['src']) + list(up_kegg_path_df['dest'])))
print('----- NUMBER OF GENES IN KEGG PATH: ' + str(len(kegg_path_gene_list)) + ' -----')
print(up_kegg_path_df.shape)

In [ ]:
# BioGRID
biogrid_df = pd.read_table('./Regulatory-network-data/BioGrid/BIOGRID-ALL-3.5.174.mitab.Symbol.txt', delimiter = '\t')
eh_list = list(biogrid_df['e_h'])
et_list = list(biogrid_df['e_t'])
# ADJUST ALL GENES TO UPPERCASE
up_eh_list = []
for eh in eh_list:
    up_eh = eh.upper()
    up_eh_list.append(up_eh)
up_et_list = []
for et in et_list:
    up_et = et.upper()
    up_et_list.append(up_et)
up_biogrid_conn_dict = {'src': up_eh_list, 'dest': up_et_list}
up_biogrid_df = pd.DataFrame(up_biogrid_conn_dict)
print(up_biogrid_df)
print(up_biogrid_df.shape)
up_biogrid_df.to_csv('./Regulatory-network-data/BioGrid/up_biogrid.csv', index = False, header = True)
up_biogrid_gene_list = list(set(list(up_biogrid_df['src']) + list(up_biogrid_df['dest'])))
print('----- NUMBER OF GENES IN BioGRID: ' + str(len(up_biogrid_gene_list)) + ' -----')

In [ ]:
# STRING
string_df = pd.read_csv('./Regulatory-network-data/STRING/9606.protein.links.detailed.v11.0_sym.csv', low_memory=False)
src_list = list(string_df['Source'])
tar_list = list(string_df['Target'])
# ADJUST ALL GENES TO UPPERCASE
up_src_list = []
for src in src_list:
    up_src = src.upper()
    up_src_list.append(up_src)
up_tar_list = []
for tar in tar_list:
    up_tar = tar.upper()
    up_tar_list.append(up_tar)
up_string_conn_dict = {'src': up_src_list, 'dest': up_tar_list}
up_string_df = pd.DataFrame(up_string_conn_dict)
print(up_string_df)
up_string_df.to_csv('./Regulatory-network-data/STRING/up_string.csv', index = False, header = True)
up_string_gene_list = list(set(list(up_string_df['src']) + list(up_string_df['dest'])))
print('----- NUMBER OF GENES IN STRING: ' + str(len(up_string_gene_list)) + ' -----')

In [ ]:
# intersect the [common genes] with the genes in the different databases [KEGG / BioGRID / STRING]
selected_database = 'KEGG'
# selected_database = 'BioGRID'
# selected_database = 'STRING'
if selected_database == 'KEGG':
    edge_common_genes = list(set(common_genes) & set(kegg_gene_list))
    print('----- NUMBER OF INTERSECTED GENES IN KEGG: ' + str(len(edge_common_genes)) + ' -----')
elif selected_database == 'BioGRID':
    edge_common_genes = list(set(common_genes) & set(up_biogrid_gene_list))
    print('----- NUMBER OF INTERSECTED GENES IN BioGRID: ' + str(len(edge_common_genes)) + ' -----')
elif selected_database == 'STRING':
    edge_common_genes = list(set(common_genes) & set(up_string_gene_list))
    print('----- NUMBER OF INTERSECTED GENES IN STRING: ' + str(len(edge_common_genes)) + ' -----')

# filter the genes in the different databases [KEGG / BioGRID / STRING] with the [common genes]
if selected_database == 'KEGG':
    filtered_up_kegg_df = up_kegg_df[up_kegg_df['src'].isin(edge_common_genes) & up_kegg_df['dest'].isin(edge_common_genes)]
    src_list = list(filtered_up_kegg_df['src'])
    dest_list = list(filtered_up_kegg_df['dest'])
    all_list = sorted(list(set(src_list + dest_list)))
    print('----- NUMBER OF INTERSECTED GENES IN KEGG: ' + str(len(all_list)) + ' -----')
    edge_common_genes = all_list
    filtered_up_kegg_df = filtered_up_kegg_df.drop_duplicates()
    filtered_up_kegg_df = filtered_up_kegg_df.sort_values(by=['src', 'dest']).reset_index(drop=True)
    print('----- NEW KEGG EDGE CONNECTIONS: ' + str(len(filtered_up_kegg_df)) + ' -----')
    filtered_up_kegg_path_df = up_kegg_path_df[up_kegg_path_df['src'].isin(edge_common_genes) & up_kegg_path_df['dest'].isin(edge_common_genes)]    
    filtered_up_kegg_path_df = filtered_up_kegg_path_df.drop_duplicates()
    filtered_up_kegg_path_df = filtered_up_kegg_path_df.sort_values(by=['src', 'dest']).reset_index(drop=True)
    print('----- NEW KEGG PATHWAY CONNECTIONS: ' + str(len(filtered_up_kegg_path_df)) filtered_up_kegg_df+ ' -----')
elif selected_database == 'BioGRID':
    filtered_up_biogrid_df = up_biogrid_df[up_biogrid_df['src'].isin(edge_common_genes) & up_biogrid_df['dest'].isin(edge_common_genes)]
    filtered_up_biogrid_df = filtered_up_biogrid_df.drop_duplicates()
    filtered_up_biogrid_df = filtered_up_biogrid_df.sort_values(by=['src', 'dest']).reset_index(drop=True)
    print('----- NEW BioGRID EDGE CONNECTIONS: ' + str(len(filtered_up_biogrid_df)) + ' -----')
elif selected_database == 'STRING':
    filtered_up_string_df = up_string_df[up_string_df['src'].isin(edge_common_genes) & up_string_df['dest'].isin(edge_common_genes)]
    filtered_up_string_df = filtered_up_string_df.drop_duplicates()
    filtered_up_string_df = filtered_up_string_df.sort_values(by=['src', 'dest']).reset_index(drop=True)
    print('----- NEW STRING EDGE CONNECTIONS: ' + str(len(filtered_up_string_df)) + ' -----')

In [ ]:
if selected_database == 'KEGG':
    display(filtered_up_kegg_df)
    display(filtered_up_kegg_path_df)
elif selected_database == 'BioGRID':
    display(filtered_up_biogrid_df)
elif selected_database == 'STRING':
    display(filtered_up_string_df)

#### 4.1.3 Filtering the gene names across the gene expression, cnv, proteomics and methylation

In [ ]:
# select common genes in gene expression data
gene_expression_filtered = gene_expression_inputted.loc[gene_expression_inputted['gene_name'].isin(edge_common_genes)]
gene_expression_filtered = gene_expression_filtered.sort_values(by=['gene_name']).reset_index(drop=True)
gene_expression_filtered

In [ ]:
protein_filtered = protein_inputted.loc[protein_inputted['gene_name'].isin(edge_common_genes)]
protein_filtered = protein_filtered.sort_values(by=['gene_name']).reset_index(drop=True)
protein_filtered

In [ ]:
# select common genes in methylation data
Upstream_df_filtered = Upstream_df_inputted.loc[Upstream_df_inputted['gene_name'].isin(edge_common_genes)]
Upstream_df_filtered = Upstream_df_filtered.sort_values(by=['gene_name']).reset_index(drop=True)
Distal_Promoter_df_filtered = Distal_Promoter_df_inputted.loc[Distal_Promoter_df_inputted['gene_name'].isin(edge_common_genes)]
Distal_Promoter_df_filtered = Distal_Promoter_df_filtered.sort_values(by=['gene_name']).reset_index(drop=True)
Proximal_Promoter_df_filtered = Proximal_Promoter_df_inputted.loc[Proximal_Promoter_df_inputted['gene_name'].isin(edge_common_genes)]
Proximal_Promoter_df_filtered = Proximal_Promoter_df_filtered.sort_values(by=['gene_name']).reset_index(drop=True)
Core_Promoter_df_filtered = Core_Promoter_df_inputted.loc[Core_Promoter_df_inputted['gene_name'].isin(edge_common_genes)]
Core_Promoter_df_filtered = Core_Promoter_df_filtered.sort_values(by=['gene_name']).reset_index(drop=True)
Downstream_df_filtered = Downstream_df_inputted.loc[Downstream_df_inputted['gene_name'].isin(edge_common_genes)]
Downstream_df_filtered = Downstream_df_filtered.sort_values(by=['gene_name']).reset_index(drop=True)

Upstream_df_filtered

In [ ]:
# select common genes in cnv data
cnv_aggregated_filtered_del = cnv_aggregated_inputted_del.loc[cnv_aggregated_inputted_del['gene_name'].isin(edge_common_genes)]
cnv_aggregated_filtered_del = cnv_aggregated_filtered_del.sort_values(by=['gene_name']).reset_index(drop=True)
cnv_aggregated_filtered_dup = cnv_aggregated_inputted_dup.loc[cnv_aggregated_inputted_dup['gene_name'].isin(edge_common_genes)]
cnv_aggregated_filtered_dup = cnv_aggregated_filtered_dup.sort_values(by=['gene_name']).reset_index(drop=True)
cnv_aggregated_filtered_mcnv = cnv_aggregated_inputted_mcnv.loc[cnv_aggregated_inputted_mcnv['gene_name'].isin(edge_common_genes)]
cnv_aggregated_filtered_mcnv = cnv_aggregated_filtered_mcnv.sort_values(by=['gene_name']).reset_index(drop=True)

cnv_aggregated_filtered_del

### 4.2 Make all data use a unified patient ID

In [ ]:
ROSMAP_biospecimen = pd.read_csv('./ROSMAP-raw/Meta-Data/ROSMAP_biospecimen_metadata.csv', sep=',')
ROSMAP_biospecimen

In [ ]:
# Split the 'specimenID' and construct the matching ID
ROSMAP_biospecimen['matching_id'] = ROSMAP_biospecimen['specimenID'].apply(lambda x: '.'.join(x.split('.')[2:4]))

# Update the dictionary mapping with the new matching IDs
matching_id_to_individual_protein = dict(zip(ROSMAP_biospecimen['matching_id'], ROSMAP_biospecimen['individualID']))

# Replace the column names in the protein DataFrame if they are in the matching_id_to_individual_protein mapping
protein_filtered.columns = [matching_id_to_individual_protein.get(col, col) for col in protein_filtered.columns]
protein_filtered

In [ ]:
mirna_ids = protein.columns.tolist()[1:]
individual_ids = protein_filtered.columns.tolist()[1:]

protein_map = pd.DataFrame({
    'individualID': individual_ids,
    'mirna_id': mirna_ids
})

protein_map

In [ ]:
matching_id_to_individual = dict(zip(ROSMAP_biospecimen['specimenID'], ROSMAP_biospecimen['individualID']))
Upstream_df_filtered.columns = [matching_id_to_individual.get(col, col) for col in Upstream_df_filtered.columns]
Distal_Promoter_df_filtered.columns = [matching_id_to_individual.get(col, col) for col in Distal_Promoter_df_filtered.columns]
Proximal_Promoter_df_filtered.columns = [matching_id_to_individual.get(col, col) for col in Proximal_Promoter_df_filtered.columns]
Core_Promoter_df_filtered.columns = [matching_id_to_individual.get(col, col) for col in Core_Promoter_df_filtered.columns]
Downstream_df_filtered.columns = [matching_id_to_individual.get(col, col) for col in Downstream_df_filtered.columns]
Upstream_df_filtered

In [ ]:
mwas_id = methylation_value.columns.tolist()[1:]
individual_ids = Upstream_df_filtered.columns.tolist()[2:]

methylation_map = pd.DataFrame({
    'individualID': individual_ids,
    'mwas_id': mwas_id
})

methylation_map

In [ ]:
def process_gene_column_name(col_name):
    parts = col_name.split('_')
    return '_'.join(parts[:2]) if len(parts) > 1 else col_name
gene_expression_filtered.columns = [process_gene_column_name(col) for col in gene_expression_filtered.columns]
matching_id_to_individual_gene = dict(zip(ROSMAP_biospecimen['specimenID'], ROSMAP_biospecimen['individualID']))
gene_expression_filtered.columns = [matching_id_to_individual_gene.get(col, col) for col in gene_expression_filtered.columns]
gene_expression_filtered

In [ ]:
mrna_id = gene_expression.columns.tolist()[1:]
individual_ids = gene_expression_filtered.columns.tolist()[1:]

gene_expression_map = pd.DataFrame({
    'individualID': individual_ids,
    'mrna_id': mrna_id
})

gene_expression_map

In [ ]:
cnv_aggregated_filtered_del.columns = [matching_id_to_individual.get(col, col) for col in cnv_aggregated_filtered_del.columns]
cnv_aggregated_filtered_dup.columns = [matching_id_to_individual.get(col, col) for col in cnv_aggregated_filtered_dup.columns]
cnv_aggregated_filtered_mcnv.columns = [matching_id_to_individual.get(col, col) for col in cnv_aggregated_filtered_mcnv.columns]
# cnv_aggregated_filtered_del

display(cnv_aggregated_filtered_del)
display(cnv_aggregated_filtered_dup)
display(cnv_aggregated_filtered_mcnv)

In [ ]:
cnvdata_id = cnv_aggregated_del.columns.tolist()[1:]
individual_ids = cnv_aggregated_filtered_del.columns.tolist()[1:]

cnv_aggregated_map = pd.DataFrame({
    'individualID': individual_ids,
    'cnvdata_id': cnvdata_id
})

cnv_aggregated_map

In [ ]:
combined_individualID = pd.concat([
    protein_map['individualID'],
    methylation_map['individualID'],
    gene_expression_map['individualID'],
    # snphead_updated_withoutNA['individualID'],
    cnv_aggregated_map['individualID'],
    survival['individualID']
]).unique()

# Now we create a new DataFrame that contains all unique 'individualID' and their corresponding values
# from the other columns in the original DataFrames.
# We perform an outer merge to ensure all unique 'individualID' are included.
union_map = pd.DataFrame(combined_individualID, columns=['individualID'])

# Merge with each original DataFrame
union_map = union_map.merge(protein_map, on='individualID', how='outer')
union_map = union_map.merge(methylation_map, on='individualID', how='outer')
union_map = union_map.merge(gene_expression_map, on='individualID', how='outer')
union_map = union_map.merge(cnv_aggregated_map, on='individualID', how='outer')
union_map = union_map.merge(survival[['individualID', 'projid', 'Study']], on='individualID', how='outer')

# Display the merged DataFrame
union_map

In [ ]:
union_map_cleaned = union_map.dropna()
union_map_cleaned.reset_index(drop=True, inplace=True)
union_map_cleaned

### 4.3 Unify patient samples within methylation, copynumer,  gene expression, clinical, proteomics, molecular subtype, and sample type, primary disease datasets

In [ ]:
#clinical data
survival

In [ ]:
#proteomics data
protein_filtered

In [ ]:
# Extract column names starting with 'R' from methylation datasets
R_columns_upstream = [col for col in Upstream_df_filtered.columns if col.startswith('R')]
R_columns_distal = [col for col in Distal_Promoter_df_filtered.columns if col.startswith('R')]
R_columns_proximal = [col for col in Proximal_Promoter_df_filtered.columns if col.startswith('R')]
R_columns_core = [col for col in Core_Promoter_df_filtered.columns if col.startswith('R')]
R_columns_downstream = [col for col in Downstream_df_filtered.columns if col.startswith('R')]
R_columns_cnv =[col for col in cnv_aggregated_filtered_del.columns if col.startswith('R')]
# Extract 'R' columns from other datasets
R_columns_gene_expression = [col for col in gene_expression_filtered.columns if col != 'gene_name']
R_columns_survival = [col for col in survival['individualID'] if col.startswith('R')]
R_columns_protein = [col for col in protein_filtered.columns if col.startswith('R')]
# Find the intersection of R column names across all DataFrames

display(R_columns_survival)

# TODO
# common_R_columns = set(R_columns_upstream) & set(R_columns_distal) & set(R_columns_proximal) & set(R_columns_core) & set(R_columns_downstream) &set(R_columns_gene_expression) &set(R_columns_survival) &set(R_columns_protein) & set(R_columns_cnv)
common_R_columns = set(R_columns_upstream) & set(R_columns_distal) & set(R_columns_proximal) & set(R_columns_core) & set(R_columns_downstream) &set(R_columns_gene_expression) &set(R_columns_survival) & set(R_columns_cnv)

# Convert the intersection back to a list, if needed
common_R_columns_list = list(common_R_columns)

# Print the number and the list of common R columns
print(f"Number of common R columns: {len(common_R_columns)}")


In [ ]:
print(R_columns_downstream[0])
print(f"Methylation Dataset has: {len(R_columns_downstream)}")
R_columns_downstream_withsurvival = set(R_columns_downstream) & set(R_columns_survival)
print(f"After intersection with survival: {len(R_columns_downstream_withsurvival)}")

print(f"Copy Number Variations Dataset has: {len(R_columns_cnv)}")
R_columns_cnv_withsurvival = set(R_columns_cnv) & set(R_columns_survival)
print(f"After intersection with survival: {len(R_columns_cnv_withsurvival)}")

print(f"RNASeq Dataset has: {len(R_columns_gene_expression)}")
R_columns_gene_expression_withsurvival = set(R_columns_gene_expression) & set(R_columns_survival)
print(f"After intersection with survival {len(R_columns_gene_expression_withsurvival)}")

# print(f"Proteomics Dataset has: {len(R_columns_protein)}")
# R_columns_protein_withsurvival = set(R_columns_protein) & set(R_columns_survival)
# print(f"After intersection with survival: {len(R_columns_protein_withsurvival)}")
# R_columns_cnv, 
# R_columns_gene_expression, 
# R_columns_survival, 
# R_columns_protein

In [ ]:
# Define columns to keep along with common R columns
additional_cols_methylation = ['gene_name']

# Filter each methylation DataFrame
Upstream_df_filtered = Upstream_df_filtered[additional_cols_methylation + common_R_columns_list]
Distal_Promoter_df_filtered = Distal_Promoter_df_filtered[additional_cols_methylation + common_R_columns_list]
Proximal_Promoter_df_filtered = Proximal_Promoter_df_filtered[additional_cols_methylation + common_R_columns_list]
Core_Promoter_df_filtered = Core_Promoter_df_filtered[additional_cols_methylation + common_R_columns_list]
Downstream_df_filtered = Downstream_df_filtered[additional_cols_methylation + common_R_columns_list]

In [ ]:
Core_Promoter_df_filtered

In [ ]:
Upstream_df_filtered

In [ ]:
# Define columns to keep along with common R columns
additional_cols_gene_expression = ['gene_name']

# Filter the gene expression DataFrame
gene_expression_filtered = gene_expression_filtered[additional_cols_gene_expression + common_R_columns_list]
gene_expression_filtered

In [ ]:
# Define columns to keep along with common R columns
additional_cols_protein = ['gene_name']

# # Filter the protein DataFrame
# protein_filtered = protein_filtered[additional_cols_protein + common_R_columns_list]
# protein_filtered

protein_filtered = protein_filtered[additional_cols_protein].copy()
for col in common_R_columns_list:
    if col not in protein_filtered.columns:
        protein_filtered[col] = 0
protein_filtered

In [ ]:
# Define columns to keep along with common R columns
additional_cols_protein = ['gene_name']

# Filter the cnv DataFrame
cnv_aggregated_filtered_del = cnv_aggregated_filtered_del[additional_cols_protein + common_R_columns_list]
cnv_aggregated_filtered_del.reset_index(drop=True, inplace=True)

cnv_aggregated_filtered_dup = cnv_aggregated_filtered_dup[additional_cols_protein + common_R_columns_list]
cnv_aggregated_filtered_dup.reset_index(drop=True, inplace=True)

cnv_aggregated_filtered_mcnv = cnv_aggregated_filtered_mcnv[additional_cols_protein + common_R_columns_list]
cnv_aggregated_filtered_mcnv.reset_index(drop=True, inplace=True)

cnv_aggregated_filtered_del

In [ ]:
survival_filtered = survival[survival['individualID'].isin(common_R_columns_list)]
survival_filtered

In [ ]:
survival_nan_column_proportions = survival_filtered.isna().mean()

# Display the results
print(survival_nan_column_proportions)

In [ ]:
# Calculate the proportion of NaN values in each column
survival_nan_column_proportions = survival_filtered.isna().mean()

# Identify columns to be dropped (where proportion of NaN values is greater than 1/3)
columns_to_drop = survival_nan_column_proportions[survival_nan_column_proportions > 1/3].index.tolist()

# Drop these columns from the DataFrame
survival_filtered = survival_filtered.drop(columns=columns_to_drop)

# List of columns that were dropped
print("Columns dropped:", columns_to_drop)

In [ ]:
survival_filtered

In [ ]:
cols = ['individualID'] + [col for col in survival_filtered.columns if col != 'individualID']
survival_filtered = survival_filtered[cols]
survival_filtered.reset_index(drop=True, inplace=True)
survival_filtered

## 5.Gene name/patient samples

In [ ]:
gene_list = gene_expression_filtered['gene_name']
gene_list

In [ ]:
patient_sample_list = pd.DataFrame(common_R_columns,columns=['sample'])
patient_sample_list

## 6.Save processed datasets

### 6.1 Keep the consistency for dataframes on genes and samples

In [ ]:
# [gene_list]
# gene-tran
sorted_gene_list = gene_list.sort_values()
sorted_gene = sorted_gene_list.tolist()
sorted_gene_tran = [gene + '-TRAN' for gene in sorted_gene]
sorted_gene_tran_df = pd.DataFrame(sorted_gene_tran, columns=['Gene'])
display(sorted_gene_tran_df)
# gene-meth
sorted_gene_methy = [gene + '-METH' for gene in sorted_gene]
sorted_gene_methy_df = pd.DataFrame(sorted_gene_methy, columns=['Gene'])
display(sorted_gene_methy_df)
# gene-protein
sorted_gene_protein = [gene + '-PROT' for gene in sorted_gene]
sorted_gene_protein_df = pd.DataFrame(sorted_gene_protein, columns=['Gene'])
display(sorted_gene_protein_df)
# all-gene
sorted_gene_all = sorted_gene_tran + sorted_gene_methy + sorted_gene_protein
sorted_all_gene_df = pd.DataFrame(sorted_gene_all, columns=['Gene'])
display(sorted_all_gene_df)

In [ ]:
# [patient-sample-list]
sorted_patient_sample_list = patient_sample_list.sort_values(by='sample')['sample'].tolist()
print(sorted_patient_sample_list)
sorted_patient_sample_df = patient_sample_list.sort_values(by='sample').reset_index(drop=True)
display(sorted_patient_sample_df)

In [ ]:
Upstream_df_filtered = Upstream_df_filtered[['gene_name'] + sorted_patient_sample_list]
Distal_Promoter_df_filtered = Distal_Promoter_df_filtered[['gene_name'] + sorted_patient_sample_list]
Proximal_Promoter_df_filtered = Proximal_Promoter_df_filtered[['gene_name'] + sorted_patient_sample_list]
Core_Promoter_df_filtered = Core_Promoter_df_filtered[['gene_name'] + sorted_patient_sample_list]
Downstream_df_filtered = Downstream_df_filtered[['gene_name'] + sorted_patient_sample_list]

Upstream_df_filtered

In [ ]:
cnv_aggregated_filtered_del = cnv_aggregated_filtered_del[['gene_name'] + sorted_patient_sample_list].sort_values(by='gene_name').reset_index(drop=True)
cnv_aggregated_filtered_dup = cnv_aggregated_filtered_dup[['gene_name'] + sorted_patient_sample_list].sort_values(by='gene_name').reset_index(drop=True)
cnv_aggregated_filtered_mcnv = cnv_aggregated_filtered_mcnv[['gene_name'] + sorted_patient_sample_list].sort_values(by='gene_name').reset_index(drop=True)

cnv_aggregated_filtered_del

In [ ]:
gene_expression_filtered = gene_expression_filtered[['gene_name'] + sorted_patient_sample_list].sort_values(by='gene_name').reset_index(drop=True)
gene_expression_filtered

In [ ]:
protein_filtered = protein_filtered[['gene_name'] + sorted_patient_sample_list].sort_values(by='gene_name').reset_index(drop=True)
protein_filtered

In [ ]:
survival_filtered = survival_filtered.sort_values(by='individualID').reset_index(drop=True)
survival_filtered

### 6.2 create output folder and save processed datasets

In [ ]:
import os

# outputfile name
output_folder = 'ROSMAP-process'
# create folder if not exist
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

In [ ]:
# DataFrame needed to be saved
dataframes = {
    'gene-tran-list.csv': sorted_gene_tran_df,
    'gene-methy-list.csv': sorted_gene_methy_df,
    'gene-protein-list.csv': sorted_gene_protein_df,
    'gene-all-list.csv': sorted_all_gene_df,
    'gene-kegg-edge-list.csv': filtered_up_kegg_df,
    'gene-kegg-path-edge-list.csv': filtered_up_kegg_path_df,
    # 'gene-biogrid-edge-list.csv': filtered_up_biogrid_df,
    # 'gene-string-edge-list.csv': filtered_up_string_df,
    'patient-sample-list.csv': sorted_patient_sample_df,
    # 'phenotype-lists.csv': phenotype_lists,
    'processed-genotype-methy-Upstream.csv': Upstream_df_filtered,
    'processed-genotype-methy-Distal-Promoter.csv': Distal_Promoter_df_filtered,
    'processed-genotype-methy-Proximal-Promoter.csv': Proximal_Promoter_df_filtered,
    'processed-genotype-methy-Core-Promoter.csv': Core_Promoter_df_filtered,
    'processed-genotype-methy-Downstream.csv': Downstream_df_filtered,
    'processed-genotype-cnv_del.csv': cnv_aggregated_filtered_del,
    'processed-genotype-cnv_dup.csv': cnv_aggregated_filtered_dup,
    'processed-genotype-cnv_mcnv.csv': cnv_aggregated_filtered_mcnv,
    'processed-genotype-gene-expression.csv': gene_expression_filtered,
    'processed-genotype-proteomics.csv': protein_filtered,
    # 'processed-phenotype-immune-subtype-transposed.csv': immune_subtype_filtered,
    'processed-phenotype-survival-transposed.csv': survival_filtered,
    # 'processed-phenotype-dense-transposed.csv': dense_filtered,
    # 'processed-phenotype-cellsub-transposed.csv': cellsub_filtered
    'processed-mapping-idmap.csv': union_map,
    'processed-mapping-idmap-withoutnull.csv': union_map_cleaned
}

# save to output folder
for file_name, df in dataframes.items():
    df.to_csv(os.path.join(output_folder, file_name), index=False)

## 7.Convert the processed data into node dictionary

In [ ]:
# load processed data
import pandas as pd
import os

# read the file names under the folder
# Define the path to the output folder where CSV files are stored
output_folder = 'ROSMAP-process'

# List of file names you saved earlier
file_names = [
    'gene-tran-list', 'gene-methy-list', 'gene-protein-list', 'gene-all-list', 
    'gene-kegg-edge-list', 'gene-kegg-path-edge-list', 
    # 'gene-biogrid-edge-list', 
    # 'gene-string-edge-list',
    'patient-sample-list', 'processed-genotype-methy-Upstream', 
    'processed-genotype-methy-Distal-Promoter', 
    'processed-genotype-methy-Proximal-Promoter', 
    'processed-genotype-methy-Core-Promoter', 'processed-genotype-methy-Downstream', 
    'processed-genotype-cnv_del', 'processed-genotype-cnv_dup', 'processed-genotype-cnv_mcnv', 
    'processed-genotype-gene-expression', 
    'processed-genotype-proteomics',
    'processed-phenotype-survival-transposed'
]

# Dictionary to hold the dataframes
dataframes = {}

# Read each file and assign to a dataframe
for file_name in file_names:
    full_path = os.path.join(output_folder, file_name + '.csv')
    dataframes[file_name] = pd.read_csv(full_path)

In [ ]:
# Assign each dataframe to a variable
sorted_gene_tran_df = dataframes['gene-tran-list']
sorted_gene_methy_df = dataframes['gene-methy-list']
sorted_gene_protein_df = dataframes['gene-protein-list']
sorted_all_gene_df = dataframes['gene-all-list']
filtered_up_kegg_df = dataframes['gene-kegg-edge-list']
filtered_up_kegg_path_df = dataframes['gene-kegg-path-edge-list']
# filtered_up_biogrid_df = dataframes['gene-biogrid-edge-list']
# filtered_up_string_df = dataframes['gene-string-edge-list']
sorted_patient_sample_df = dataframes['patient-sample-list']
# phenotype_lists = dataframes['phenotype-lists']
Upstream_df_filtered = dataframes['processed-genotype-methy-Upstream']
Distal_Promoter_df_filtered = dataframes['processed-genotype-methy-Distal-Promoter']
Proximal_Promoter_df_filtered = dataframes['processed-genotype-methy-Proximal-Promoter']
Core_Promoter_df_filtered = dataframes['processed-genotype-methy-Core-Promoter']
Downstream_df_filtered = dataframes['processed-genotype-methy-Downstream']
copynumber_filtered_del = dataframes['processed-genotype-cnv_del']
copynumber_filtered_dup = dataframes['processed-genotype-cnv_dup']
copynumber_filtered_mcnv = dataframes['processed-genotype-cnv_mcnv']
gene_expression_filtered = dataframes['processed-genotype-gene-expression']
protein_filtered = dataframes['processed-genotype-proteomics']
# immune_subtype_filtered = dataframes['processed-phenotype-immune-subtype-transposed']
survival_filtered = dataframes['processed-phenotype-survival-transposed']
# dense_filtered = dataframes['processed-phenotype-dense-transposed']
# cellsub_filtered = dataframes['processed-phenotype-cellsub-transposed']

In [ ]:
# outputfile name
graph_output_folder = 'ROSMAP-graph-data'
# create folder if not exist
if not os.path.exists(graph_output_folder):
    os.makedirs(graph_output_folder)

### 7.1 Make nodes dictionary

In [ ]:
sorted_all_gene_dict = sorted_all_gene_df['Gene'].to_dict()
sorted_all_gene_name_dict = {value: key for key, value in sorted_all_gene_dict.items()}
num_gene = sorted_gene_tran_df.shape[0]
num_gene_protein = sorted_gene_protein_df.shape[0]
nodetype_list = ['Gene-TRAN'] * num_gene + ['Gene-METH'] * num_gene + ['Gene-PROT'] * num_gene_protein
map_all_gene_df = pd.DataFrame({'Gene_num': sorted_all_gene_dict.keys(), 'Gene_name': sorted_all_gene_dict.values(), 'NodeType': nodetype_list})
display(map_all_gene_df)
map_all_gene_df.to_csv(os.path.join(graph_output_folder, 'map-all-gene.csv'), index=False)

### 7.2 Create the edges connection between promoter methylations and proteins

In [ ]:
# [Gene-METH - Gene]
sorted_gene_methy = sorted_gene_methy_df['Gene'].tolist()
sorted_gene_list = sorted_gene_tran_df['Gene'].tolist()
sorted_gene_protein = sorted_gene_protein_df['Gene'].tolist()
sorted_intersection = [gene_protein.replace('-PROT', '-TRAN') for gene_protein in sorted_gene_protein]
gene_meth_edge_df = pd.DataFrame({'src': sorted_gene_methy, 'dest': sorted_gene_list})
display(gene_meth_edge_df)
# [Gene - Gene-PROT]
gene_protein_edge_df = pd.DataFrame({'src': sorted_intersection, 'dest': sorted_gene_protein})
display(gene_protein_edge_df)

In [ ]:
print(sorted_all_gene_name_dict['ABL1-TRAN'])
print(sorted_all_gene_name_dict['ABL1-METH'])
print(sorted_all_gene_name_dict['ABL1-PROT'])

In [ ]:
# replace gene name with gene number
gene_meth_num_edge_df = gene_meth_edge_df.copy()
gene_meth_num_edge_df['src'] = gene_meth_edge_df['src'].map(sorted_all_gene_name_dict)
gene_meth_num_edge_df['dest'] = gene_meth_edge_df['dest'].map(sorted_all_gene_name_dict)
display(gene_meth_num_edge_df)
gene_protein_num_edge_df = gene_protein_edge_df.copy()
gene_protein_num_edge_df['src'] = gene_protein_edge_df['src'].map(sorted_all_gene_name_dict)
gene_protein_num_edge_df['dest'] = gene_protein_edge_df['dest'].map(sorted_all_gene_name_dict)
display(gene_protein_num_edge_df)

### 7.3 Concat all edges

In [ ]:
if selected_database == 'KEGG':
    # filtered_up_num_df = filtered_up_kegg_df.copy()
    filtered_up_num_df = filtered_up_kegg_path_df.copy()
elif selected_database == 'BioGRID':
    filtered_up_num_df = filtered_up_biogrid_df.copy()
elif selected_database == 'STRING':
    filtered_up_num_df = filtered_up_string_df.copy()

# Add 'PROT' to the end of each gene name in the 'src' and 'dest' columns
filtered_up_num_df['src'] = filtered_up_num_df['src'].apply(lambda x: x + '-PROT')
filtered_up_num_df['dest'] = filtered_up_num_df['dest'].apply(lambda x: x + '-PROT')

filtered_up_num_df['src'] = filtered_up_num_df['src'].map(sorted_all_gene_name_dict)
filtered_up_num_df['dest'] = filtered_up_num_df['dest'].map(sorted_all_gene_name_dict)
display(filtered_up_num_df)
all_gene_edge_num_df = pd.concat([filtered_up_num_df, gene_meth_num_edge_df, gene_protein_num_edge_df])
display(all_gene_edge_num_df)

num_gene_edge = filtered_up_num_df.shape[0]
num_gene_meth_edge = gene_meth_num_edge_df.shape[0]
num_gene_protein_edge = gene_protein_num_edge_df.shape[0]
edgetype_list = ['Gene-PROT-Gene-PROT'] * num_gene_edge + ['Gene-TRAN-Gene-METH'] * num_gene_meth_edge + ['Gene-TRAN-Gene-PROT'] * num_gene_protein_edge
all_gene_edge_num_df['EdgeType'] = edgetype_list
all_gene_edge_num_df = all_gene_edge_num_df.sort_values(by=['src', 'dest']).reset_index(drop=True)
all_gene_edge_num_df.fillna('internal link', inplace=True)
display(all_gene_edge_num_df)
all_gene_edge_num_df.to_csv(os.path.join(graph_output_folder, 'all-gene-edge-num.csv'), index=False)

In [ ]:
# gene edge interactions without map
all_gene_edge_df = all_gene_edge_num_df.copy()
all_gene_edge_df = all_gene_edge_df.replace(sorted_all_gene_dict)

num_gene_edge = filtered_up_num_df.shape[0]
num_gene_meth_edge = gene_meth_edge_df.shape[0]
num_gene_protein_edge = gene_protein_edge_df.shape[0]
# all_gene_edge_df = all_gene_edge_df.sort_values(by=['src', 'dest']).reset_index(drop=True)
all_gene_edge_df.to_csv(os.path.join(graph_output_folder, 'all-gene-edge.csv'), index=False)
display(all_gene_edge_df)

## 8.Load data into graph format

### 8.1 Form up the input samples

recommends the use of the ceradsc as the classfication for AD types

* ceradsc
* cogdx

#### Balance data

In [ ]:
# Change it to binary
def modify_ceradsc(value):
    if value in [1, 2]:
        return 0
    elif value in [3, 4]:
        return 1
    else:
        return value  # Keeps other values as they are, if there are any

survival_filtered['ceradsc'] = survival_filtered['ceradsc'].apply(modify_ceradsc)
survival_filtered

In [ ]:
survival_filtered

count_female = (survival_filtered['msex'] == 0).sum()
count_male = (survival_filtered['msex'] == 1).sum()
print(count_female, count_male)

count_AD = ((survival_filtered['ceradsc'] == 0)).sum()
count_NOAD = ((survival_filtered['ceradsc'] == 1)).sum()
count_AD, count_NOAD
print(count_AD, count_NOAD)

count_female_AD = ((survival_filtered['msex'] == 0) & ((survival_filtered['ceradsc'] == 0))).sum()
count_male_AD = ((survival_filtered['msex'] == 1) & ((survival_filtered['ceradsc'] == 0))).sum()
print('AD Female:', count_female_AD)
print('AD Male:', count_male_AD)

In [ ]:
from sklearn.utils import resample

# Calculate the number of AD and non-AD samples in the original dataset
count_AD = (((survival_filtered['ceradsc'] == 0)) ).sum()
count_NOAD = ((survival_filtered['ceradsc'] == 1) ).sum()

# Get the AD and non-AD datasets
df_AD = survival_filtered[((survival_filtered['ceradsc'] == 0))]
df_NOAD = survival_filtered[(survival_filtered['ceradsc'] == 1)]

# Determine the number of samples after downsampling, taking the smaller value between female and male AD sample counts
n_samples = min(count_AD, count_NOAD)

# Downsample the AD dataset
df_AD_downsampled = resample(df_AD, 
                                    replace=False,
                                    n_samples=n_samples,
                                    random_state=123)

# Downsample the non-AD dataset
df_NOAD_downsampled = resample(df_NOAD, 
                                  replace=False,
                                  n_samples=n_samples,
                                  random_state=123)

# Combine the downsampled AD and non-AD datasets
df_balanced_ds = pd.concat([df_AD_downsampled, df_NOAD_downsampled]).reset_index(drop=True)

# Print the counts of the downsampled female and male AD samples
count_AD_downsampled = len(df_AD_downsampled)
count_NOAD_downsampled = len(df_NOAD_downsampled)
print('AD:', count_AD_downsampled)
print('non-AD:', count_NOAD_downsampled)

# Overwrite the original survival_filtered with the balanced dataset
survival_filtered = df_balanced_ds
survival_filtered


In [ ]:
# Filter the dataset to keep only AD samples
df_AD = survival_filtered[((survival_filtered['ceradsc'] == 0))]
df_NOAD = survival_filtered[(survival_filtered['ceradsc'] == 1)]

# Calculate the number of female and male AD samples
count_female_AD = (df_AD['msex'] == 0).sum()
count_male_AD = (df_AD['msex'] == 1).sum()

# Print the counts of female and male AD samples
print('AD Female:', count_female_AD)
print('AD Male:', count_male_AD)

# Calculate the number of female and male NOAD samples
count_female_NOAD = (df_NOAD['msex'] == 0).sum()
count_male_NOAD = (df_NOAD['msex'] == 1).sum()

# Print the counts of female and male NOAD samples
print('non-AD Female:',  count_female_NOAD)
print('non-AD Male:', count_male_NOAD)

# Overwrite the original survival_filtered with the balanced dataset
survival_filtered = df_balanced_ds.reset_index(drop=True)
survival_filtered


In [ ]:
survival_filtered_feature_df = survival_filtered.copy()

# TODO
survival_filtered_feature_df = survival_filtered_feature_df[['individualID', 'ceradsc']]
# survival_filtered_feature_df = survival_filtered_feature_df[['individualID', 'msex']]
display(survival_filtered_feature_df)

nan_counts = survival_filtered_feature_df.isna().sum()  # or df.isnull()

# TODO
print(survival_filtered_feature_df['ceradsc'].unique())
# print(survival_filtered_feature_df['msex'].unique())

survival_filtered_feature_df.to_csv(os.path.join(graph_output_folder, 'survival-label.csv'), index=False)

### 8.2 Randomize the input label

In [ ]:
# Randomize the survival label
def input_random(randomized, graph_output_folder):
    if randomized == True:
        random_survival_filtered_feature_df = survival_filtered_feature_df.sample(frac = 1).reset_index(drop=True)
        random_survival_filtered_feature_df.to_csv(os.path.join(graph_output_folder, 'random-survival-label.csv'), index=False)
    else:
        random_survival_filtered_feature_df = pd.read_csv(os.path.join(graph_output_folder, 'random-survival-label.csv'))
    display(random_survival_filtered_feature_df)

# TODO
# input_random(randomized=True, graph_output_folder=graph_output_folder)
input_random(randomized=False, graph_output_folder=graph_output_folder)

### 8.3 Split the randomized input into K-fold

In [ ]:
# Split deep learning input into training and test
def split_k_fold(k, graph_output_folder):
    random_survival_filtered_feature_df = pd.read_csv(os.path.join(graph_output_folder, 'random-survival-label.csv'))
    num_points = random_survival_filtered_feature_df.shape[0]
    num_div = int(num_points / k)
    num_div_list = [i * num_div for i in range(0, k)]
    num_div_list.append(num_points)
    # Split [random_survival_filtered_feature_df] into [k] folds
    for place_num in range(k):
        low_idx = num_div_list[place_num]
        high_idx = num_div_list[place_num + 1]
        print('\n--------TRAIN-TEST SPLIT WITH TEST FROM ' + str(low_idx) + ' TO ' + str(high_idx) + '--------')
        split_input_df = random_survival_filtered_feature_df[low_idx : high_idx]
        split_input_df.to_csv(os.path.join(graph_output_folder, 'split-random-survival-label-' + str(place_num + 1) + '.csv'), index=False)
        print(split_input_df.shape)

split_k_fold(k=5, graph_output_folder=graph_output_folder)

### 8.4 Reprocess the edge_index file

In [ ]:
import os
import numpy as np
import pandas as pd

graph_output_folder = 'ROSMAP-graph-data'
gene_edge_num_df = pd.read_csv(os.path.join(graph_output_folder, 'all-gene-edge-num.csv'))
src_gene_list = list(gene_edge_num_df['src'])
dest_gene_list = list(gene_edge_num_df['dest'])
edgetype_list = list(gene_edge_num_df['EdgeType'])
path_list = list(gene_edge_num_df['path'])
gene_edge_num_reverse_df = pd.DataFrame({'src': dest_gene_list, 'dest': src_gene_list, 'path': path_list, 'EdgeType': edgetype_list})
gene_edge_num_reverse_wointernal_df = gene_edge_num_reverse_df[~gene_edge_num_reverse_df['path'].str.contains('internal link')]
display(gene_edge_num_df)
display(gene_edge_num_reverse_wointernal_df)
gene_edge_num_all_df = pd.concat([gene_edge_num_df, gene_edge_num_reverse_wointernal_df]).drop_duplicates().sort_values(by=['src', 'dest']).reset_index(drop=True)
display(gene_edge_num_all_df)


In [ ]:
display(gene_edge_num_all_df)
gene_edge_num_all_df.to_csv(os.path.join(graph_output_folder, 'gene_edge_num_all_df.csv'), index=False)

gene_edge_name_all_df = gene_edge_num_all_df.replace(sorted_all_gene_dict)
display(gene_edge_name_all_df)
gene_edge_name_all_df.to_csv(os.path.join(graph_output_folder, 'gene_edge_name_all_df.csv'), index=False)

In [ ]:
# remove internal link
kegg_path_gene_edge_num_all_df = gene_edge_num_all_df[~gene_edge_num_all_df['path'].str.contains('internal link')]
display(kegg_path_gene_edge_num_all_df)
kegg_path_gene_edge_num_all_df.to_csv(os.path.join(graph_output_folder, 'keggpath-gene-edge-num-all.csv'), index=False)
# keep only internal link
internal_gene_edge_num_all_df = gene_edge_num_all_df[gene_edge_num_all_df['path'].str.contains('internal link')]
display(internal_gene_edge_num_all_df)
internal_gene_edge_num_all_df.to_csv(os.path.join(graph_output_folder, 'internal-gene-edge-num-all.csv'), index=False)

kegg_path_gene_edge_name_all_df = kegg_path_gene_edge_num_all_df.replace(sorted_all_gene_dict)
kegg_path_gene_edge_name_all_df.to_csv(os.path.join(graph_output_folder, 'keggpath-gene-edge-name-all.csv'), index=False)
internal_gene_edge_name_all_df = internal_gene_edge_num_all_df.replace(sorted_all_gene_dict)
internal_gene_edge_name_all_df.to_csv(os.path.join(graph_output_folder, 'internal-gene-edge-name-all.csv'), index=False)